In [ ]:
import numpy as np
import emcee
import matplotlib.pyplot as plt
import _fig_params
from scipy.optimize import minimize

np.random.seed(3)

# Markov chain Monte Carlo

First, lets generate some fake data.

In [ ]:
x = np.linspace(1, 9, 5)
y = 1.123 * x + 2.63 + (np.random.randn(5) * 0.5)
dy = np.ones(5)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(x, y, dy, marker='o', ls='')
ax.set_xlabel('$x$')
ax.set_ylabel('$y$')
ax.set_ylim(0, 16)
ax.set_xlim(0, 10)
plt.tight_layout()

The model that we want to use is that of a straight line, and the log likelihood of this is found using the function below. 

In [ ]:
def straight_line(x, m, c):
    return m * x + c

In [ ]:
def log_likelihood(theta, x, y, dy):
    m, c = theta
    model = straight_line(x, m, c)
    sigma2 = dy ** 2
    return -0.5 * np.sum((y - model) ** 2 / sigma2 + np.log(sigma2))

We maximise the log_likelihood with a negative minimisation.

In [ ]:
nll = lambda *args: -log_likelihood(*args)
initial = np.array([1, 3]) + 0.1 * np.random.randn(2)
soln = minimize(nll, initial, args=(x, y, dy))
m_ml, c_ml = soln.x

In [ ]:
m_ml, c_ml

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(x, y, dy, marker='o', ls='')
ax.plot(x, straight_line(x, m_ml, c_ml))
ax.set_xlabel('$x$')
ax.set_ylabel('$y$')
ax.set_ylim(0, 16)
ax.set_xlim(0, 10)
plt.tight_layout()

Having found a good solution, we can then leverage MCMC.

In [ ]:
pos = soln.x + 1e-1 * np.random.randn(32, 2)
nwalkers, ndim = pos.shape

sampler = emcee.EnsembleSampler(nwalkers, ndim, log_likelihood, args=(x, y, dy))
sampler.run_mcmc(pos, 5000, progress=True);

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
hist = ax.hist2d(sampler.flatchain[:, 0], sampler.flatchain[:, 1], 
                 bins=100, cmap="Blues", density=True)
ax.set_xlim(0.6, 1.3)
ax.set_ylim(1.6, 5.3)
ax.set_xticks(np.percentile(sampler.flatchain[:, 0], [2.5, 50, 97.5]))
ax.set_yticks(np.percentile(sampler.flatchain[:, 1], [2.5, 50, 97.5]))
ax.set_xlabel('$m$')
ax.set_ylabel('$c$')
plt.show()